# 03 - Monitoring ExperimentsRun data drift, concept drift, and performance monitoring on production data.- Run full monitoring cycle- Inspect drift reports- Visualize drift statistics and performance alerts

In [ ]:
import jsonimport pandas as pdimport plotly.express as pxfrom pathlib import Pathfrom src.monitoring.monitoring_report import run_monitoring_cyclefrom src.data.data_loader import load_config, load_reference_data, load_production_data, load_predictions_logfrom src.monitoring.data_drift import detect_data_driftfrom src.monitoring.performance_monitor import rolling_performancefrom src.monitoring.concept_drift import detect_concept_driftconfig = load_config()print("Configuration loaded.")

In [ ]:
results = run_monitoring_cycle(config=config)print(f"Monitoring cycle completed at: {results['timestamp']}")print()print("=== DATA DRIFT ===")print(f"Overall drifted: {results['data_drift']['overall_drifted']}")print(f"Drifted features: {results['data_drift']['drifted_features']}")print()print("=== CONCEPT DRIFT ===")print(f"N predictions: {results['concept_drift']['n_predictions']}")print(f"Overall drifted: {results['concept_drift']['overall_drifted']}")print()print("=== PERFORMANCE ===")print(f"Overall passed: {results['performance']['overall_passed']}")print(f"N alerts: {results['performance']['n_alerts']}")

In [ ]:
reference = load_reference_data(config)production = load_production_data(config)drift_report = detect_data_drift(reference, production, config=config)drift_rows = []for f in drift_report.feature_results:    drift_rows.append({        "feature": f.feature,        "type": f.drift_type,        "statistic": round(f.statistic, 4),        "p_value": round(f.p_value, 4),        "psi": round(f.psi_value, 4),        "drifted": f.drifted,    })drift_df = pd.DataFrame(drift_rows)display(drift_df)

In [ ]:
drifted = drift_df[drift_df["drifted"] == True]if not drifted.empty:    fig = px.bar(        drift_df,        x="feature",        y="psi",        color="drifted",        color_discrete_map={True: "#E4572E", False: "#2E86AB"},        title="Population Stability Index (PSI) by Feature",        labels={"psi": "PSI Value", "feature": "Feature"},    )    fig.add_hline(y=0.2, line_dash="dash", line_color="red", annotation_text="PSI Threshold (0.2)")    fig.show()else:    print("No features exceeded the PSI drift threshold.")

In [ ]:
predictions = load_predictions_log(config)print(f"Prediction log shape: {predictions.shape}")print(predictions.head())perf_cfg = config.get("performance", {})window_size = perf_cfg.get("window_size", 100)labeled = predictions.dropna(subset=["actual_label", "prediction"])if not labeled.empty:    perf_report = rolling_performance(labeled, window_size=window_size, config=config)    print(f"Windows evaluated: {perf_report.n_windows}")    print(f"Overall passed: {perf_report.overall_passed}")    print(f"Alerts: {len(perf_report.alerts)}")    for alert in perf_report.alerts[:5]:        print(f"  [{alert.severity}] {alert.metric}: {alert.current_value:.4f} < {alert.threshold}")else:    print("No labeled predictions available for performance monitoring.")

## Key Observations- Data drift is detected on features with shifted distributions.- Performance metrics are computed over rolling windows and compared against thresholds.- Reports are written as JSON files to `drift_reports/` and `performance_reports/`.- The full monitoring cycle can be re-run via `python app.py --mode monitor`.